# RAG 第 9 课：完整 Pipeline 入门

前面几节我们把 RAG 的关键模块拆开学了：
- chunking
- embeddings
- reranking
- hybrid search
- retrieval evaluation
- generation evaluation
- eval set 与 error analysis

这节课我们第一次把这些想法串起来，搭一个更像真实项目的教学版 pipeline：

1. `query rewrite`
2. `retrieve`
3. `rerank`
4. `generate`
5. `evaluate`

它依然是教学版，不依赖外部 API，但整体结构已经很接近真实系统。

In [ ]:
# 先清理 GPU 缓存
import gc

try:
    import torch
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception as e:
    print(f'Torch not available, skip GPU cache clear. ({e})')

## 1. 知识库与测试样本

我们准备一个小型知识库和几条测试问题。

In [ ]:
import re
from collections import Counter

kb = [
    'E401 means authentication failed because the API token is invalid or expired.',
    'Reset your password by opening Settings, then Security, then Reset Password.',
    'Chunking splits long documents into smaller passages to improve retrieval focus.',
    'Embeddings convert text into vectors so semantic similarity can be measured.',
    'Hybrid search combines lexical matching and dense retrieval scores.',
    'Reranking reorders retrieved candidates so the strongest evidence appears first.',
    'Billing reports can be exported from the Reports page as CSV files.',
]

dataset = [
    {
        'query': 'Why am I seeing auth failure E401?',
        'gold_doc_id': 0,
        'reference_answer': 'E401 means authentication failed because the token is invalid or expired.',
    },
    {
        'query': 'How do I change my password?',
        'gold_doc_id': 1,
        'reference_answer': 'Open Settings, go to Security, and choose Reset Password.',
    },
    {
        'query': 'Why does RAG use chunking?',
        'gold_doc_id': 2,
        'reference_answer': 'Chunking splits long documents into smaller passages so retrieval becomes more focused.',
    },
    {
        'query': 'How can I download billing reports?',
        'gold_doc_id': 6,
        'reference_answer': 'You can export billing reports from the Reports page as CSV files.',
    },
]

for i, doc in enumerate(kb):
    print(f'doc {i}: {doc}')

## 2. 一些基础工具

这里我们继续用最小实现，方便把注意力放在 pipeline 本身。

In [ ]:
def normalize_token(token):
    token = token.lower()
    token = re.sub(r'[^a-z0-9]+', '', token)
    if token.endswith('ing') and len(token) > 5:
        token = token[:-3]
    elif token.endswith('ed') and len(token) > 4:
        token = token[:-2]
    elif token.endswith('s') and len(token) > 4:
        token = token[:-1]
    return token


def tokenize(text):
    return [t for t in (normalize_token(x) for x in text.split()) if t]


def normalize_text(text):
    return ' '.join(tokenize(text))

## 3. 第一步：Query Rewrite

真实系统里，用户输入经常很口语化、很含糊。
所以很多 RAG pipeline 会先做一次 query rewrite，把问题改写得更适合检索。

这里我们做一个教学版 rewrite：
- 把口语表达改成知识库更容易命中的写法
- 补一点同义词映射

In [ ]:
def rewrite_query(query):
    rewritten = query.lower()
    replacements = {
        'auth failure': 'authentication failed',
        'change my password': 'reset password',
        'download billing reports': 'export billing reports csv',
        'why does rag use chunking': 'chunking improve retrieval focus',
    }
    for src, dst in replacements.items():
        rewritten = rewritten.replace(src, dst)
    return rewritten


for item in dataset:
    print('original :', item['query'])
    print('rewritten:', rewrite_query(item['query']))
    print()

## 4. 第二步：Retrieve

这一版先用一个非常简单的 lexical retrieval。
真实系统里这里可能会换成 BM25、embedding retrieval、hybrid search。

In [ ]:
def lexical_score(query, text):
    q_tokens = tokenize(query)
    d_tokens = set(tokenize(text))
    score = 0.0
    for token in q_tokens:
        if token in d_tokens:
            score += 1.0
            if any(ch.isdigit() for ch in token):
                score += 1.5
    return score


def retrieve(query, top_k=3):
    rows = []
    for doc_id, doc in enumerate(kb):
        rows.append({
            'doc_id': doc_id,
            'score': lexical_score(query, doc),
            'text': doc,
        })
    rows.sort(key=lambda x: x['score'], reverse=True)
    return rows[:top_k]

## 5. 第三步：Rerank

retrieval 先做粗召回，rerank 再做精排。
这里我们做一个很简化的 reranker：
- 优先奖励更完整覆盖 query 关键词的文档
- 对某些关键短语做额外加分

In [ ]:
def rerank(query, candidates):
    q_tokens = set(tokenize(query))
    reranked = []
    for row in candidates:
        doc_tokens = set(tokenize(row['text']))
        overlap = len(q_tokens & doc_tokens)
        bonus = 0.0

        lowered = row['text'].lower()
        if 'csv' in query.lower() and 'csv' in lowered:
            bonus += 1.0
        if 'authentication' in query.lower() and 'authentication failed' in lowered:
            bonus += 1.0
        if 'reset password' in query.lower() and 'reset password' in lowered:
            bonus += 1.0

        rerank_score = overlap + bonus + 0.1 * row['score']
        reranked.append({**row, 'rerank_score': rerank_score})

    reranked.sort(key=lambda x: x['rerank_score'], reverse=True)
    return reranked

## 6. 第四步：Generate

这一格我们仍然用一个教学版生成器。
重点不是生成模型本身有多强，而是看完整 pipeline 的数据流。

In [ ]:
def generate_answer(query, context):
    q = query.lower()
    c = context.lower()

    if 'e401' in q:
        if 'e401' in c:
            return 'E401 means authentication failed because the token is invalid or expired.'
        return 'E401 is related to a timeout issue.'

    if 'password' in q:
        if 'reset password' in c:
            return 'Open Settings, go to Security, and choose Reset Password.'
        return 'You need to contact support to change your password.'

    if 'chunking' in q:
        if 'chunking' in c:
            return 'Chunking splits long documents into smaller passages so retrieval becomes more focused.'
        return 'Chunking mainly reduces storage size.'

    if 'billing reports' in q or 'report' in q:
        if 'csv' in c or 'reports page' in c:
            return 'You can export billing reports from the Reports page as CSV files.'
        return 'Billing reports cannot be exported.'

    return 'I do not know.'

## 7. 把完整 pipeline 串起来

现在我们把前面几步整合成一个统一函数。

In [ ]:
def rag_pipeline(query, top_k=3):
    rewritten_query = rewrite_query(query)
    candidates = retrieve(rewritten_query, top_k=top_k)
    reranked = rerank(rewritten_query, candidates)
    best = reranked[0]
    answer = generate_answer(query, best['text'])
    return {
        'original_query': query,
        'rewritten_query': rewritten_query,
        'retrieved': candidates,
        'reranked': reranked,
        'chosen_doc_id': best['doc_id'],
        'chosen_context': best['text'],
        'answer': answer,
    }

In [ ]:
demo = rag_pipeline(dataset[0]['query'])
print('original_query :', demo['original_query'])
print('rewritten_query:', demo['rewritten_query'])
print('\nretrieved top-3:')
for row in demo['retrieved']:
    print(f"doc={row['doc_id']} | score={row['score']:.3f}")
    print(row['text'])
    print()

print('reranked top-3:')
for row in demo['reranked']:
    print(f"doc={row['doc_id']} | rerank_score={row['rerank_score']:.3f}")
    print(row['text'])
    print()

print('final answer:')
print(demo['answer'])

## 8. 对完整 pipeline 做简单评估

这一课我们继续用最简单、最直观的一组指标：
- `Retrieval Hit@1`
- `Exact Match`
- `Token F1`

这样你可以同时看到：
- 检索选得对不对
- 最终答案对不对

In [ ]:
def exact_match(prediction, reference):
    return 1.0 if normalize_text(prediction) == normalize_text(reference) else 0.0


def token_f1(prediction, reference):
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)
    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)
    overlap = sum((pred_counter & ref_counter).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

In [ ]:
def evaluate_pipeline(dataset):
    rows = []
    retrieval_scores = []
    em_scores = []
    f1_scores = []

    for item in dataset:
        result = rag_pipeline(item['query'])
        retrieval_hit = 1.0 if result['chosen_doc_id'] == item['gold_doc_id'] else 0.0
        em = exact_match(result['answer'], item['reference_answer'])
        f1 = token_f1(result['answer'], item['reference_answer'])

        retrieval_scores.append(retrieval_hit)
        em_scores.append(em)
        f1_scores.append(f1)

        rows.append({
            'query': item['query'],
            'rewritten_query': result['rewritten_query'],
            'gold_doc_id': item['gold_doc_id'],
            'chosen_doc_id': result['chosen_doc_id'],
            'retrieval_hit': retrieval_hit,
            'answer': result['answer'],
            'reference_answer': item['reference_answer'],
            'exact_match': em,
            'token_f1': f1,
        })

    summary = {
        'RetrievalHit@1': sum(retrieval_scores) / len(retrieval_scores),
        'ExactMatch': sum(em_scores) / len(em_scores),
        'TokenF1': sum(f1_scores) / len(f1_scores),
    }
    return summary, rows

In [ ]:
summary, rows = evaluate_pipeline(dataset)
print(summary)

## 9. 看每条样本的中间结果

这一步很接近真实调试流程。
你会发现 pipeline 的价值，不只是结果更好，还在于问题定位更容易。

In [ ]:
for row in rows:
    print('query:', row['query'])
    print('rewritten_query:', row['rewritten_query'])
    print('gold_doc_id:', row['gold_doc_id'], '| chosen_doc_id:', row['chosen_doc_id'], '| retrieval_hit:', row['retrieval_hit'])
    print('answer:', row['answer'])
    print('reference:', row['reference_answer'])
    print('exact_match:', row['exact_match'], '| token_f1:', round(row['token_f1'], 3))
    print('-' * 90)

## 10. 这节课最重要的 takeaway

你现在应该能看到，真实 RAG 往往不是“用户问题直接去搜文档”这么简单，而更像：

- 先把 query 变得更适合检索
- 再做一轮召回
- 再做一轮精排
- 最后才生成答案

也正因为有这些中间层，系统才更容易调试和优化。

## 11. 本课小结

这节课你要带走的核心是：

1. 一个更完整的 RAG pipeline 往往包含 `rewrite -> retrieve -> rerank -> generate`。
2. Query rewrite 在真实系统里很常见，因为用户输入并不总适合直接检索。
3. 有了完整 pipeline，后续 evaluation 和 error analysis 会更有抓手。

下一课最自然的衔接就是：
**如何比较 pipeline 改动前后到底有没有变好，也就是 A/B 对比与实验设计。**